# FYP Phase 2 — Ultimate Restormer (one model for blur Levels 5–10)

**What changes compared with the per-level runs**

| Problem seen in Level 5 / 6 | Fix in this notebook |
|---|---|
| Blobs, dots and holes inside letters (model guesses because a 128 px patch is smaller than the blur) | Progressive patches 160 → 224 → 288 px |
| Validation flat after ~1000 iters, frozen encoder can't adapt to heavier blur | Full fine-tuning (encoder unfrozen, lower LR for encoder) |
| Only 150 training images per level | One model trained on all 6 levels (~900 images) |
| One global affine per level; alignment differs between levels | Per-level affine + per-image ECC refinement |
| Pixel loss only | Charbonnier + FFT (edge sharpness) + SSIM loss |
| Noisy checkpoint selection | EMA weights, 60 validation images, optional 4-flip test-time augmentation |

Everything is saved in `FinalYearProject-Phase2/ultimate_model/`. Training is resumable: if Colab disconnects, just *Run all* again.

**Runtime:** use the fastest GPU you have (A100 > L4 > T4). The training cell prints the time per iteration; reduce `STAGES` if it is too slow.

In [ ]:
# ============================================================
# CONFIG
# ============================================================
LEVELS        = [5, 6, 7, 8, 9, 10]
RESUME        = True      # continue training from ultimate_last.pth if it exists
USE_TTA       = True      # also evaluate with 4-flip test-time augmentation
SAVE_RESTORED = True      # save restored test images (16-bit PNG) for the ultimate model

from google.colab import drive
drive.mount('/content/drive')

import os, glob, json, shutil, subprocess, re, random, time, math

PHASE2     = '/content/drive/MyDrive/FinalYearProject-Phase2'
PRETRAINED = f'{PHASE2}/pretrained/single_image_defocus_deblurring.pth'

ULT       = f'{PHASE2}/ultimate_model'
ULT_CKPT  = f'{ULT}/checkpoints'
ULT_RES   = f'{ULT}/results'
ULT_LOG   = f'{ULT}/logs'
ULT_FIG   = f'{ULT}/results/figures'
ULT_REST  = f'{ULT}/results/restored'
ULT_CACHE = f'{ULT}/cache'          # registration matrices, so a restart skips re-matching
LOCAL     = '/content/ultimate_data' # preprocessed images (local disk, fast)

for d in [ULT_CKPT, ULT_RES, ULT_LOG, ULT_FIG, ULT_REST, ULT_CACHE, f'{ULT_RES}/per_level',
          f'{ULT_RES}/splits', f'{ULT_RES}/registration', LOCAL]:
    os.makedirs(d, exist_ok=True)

assert os.path.exists(PRETRAINED), f'Pretrained weights not found: {PRETRAINED}'


def level_ckpt(level, kind):
    """levelN_best/last.pth of a per-level run (also accepts the old 'checkpoints__' prefix)."""
    folder = f'{PHASE2}/level_{level:02d}/checkpoints'
    clean_name = f'{folder}/level{level}_{kind}.pth'
    old_name = f'{folder}/checkpoints__level{level}_{kind}.pth'
    if os.path.exists(clean_name):
        return clean_name
    if os.path.exists(old_name):
        return old_name
    return None


def level_result_file(level, name):
    base = f'{PHASE2}/level_{level:02d}/results'
    for p in [f'{base}/level{level}_results/{name}', f'{base}/{name}']:
        if os.path.exists(p):
            return p
    return None


def find_zip(level):
    data_dir = f'{PHASE2}/level_{level:02d}/dataset'
    expected = f'{data_dir}/HDC2021_step{level}.zip'
    if os.path.exists(expected):
        return expected
    zips = sorted(glob.glob(f'{data_dir}/*.zip'))
    if len(zips) > 0:
        return zips[0]
    return None


missing = []
for lv in LEVELS:
    z = find_zip(lv)
    if z is None:
        missing.append(lv)
    else:
        print(f'Level {lv:2d}: {os.path.basename(z)}  ({os.path.getsize(z) / 1e9:.2f} GB)  '
              f'| per-level model: {level_ckpt(lv, "best")}')
assert len(missing) == 0, f'Dataset zip missing for levels {missing}'

total, used, free = shutil.disk_usage('/content')
print(f'\nLocal disk free: {free / 1e9:.0f} GB (needs ~25 GB)')
print('Outputs ->', ULT)

In [ ]:
!pip install -q einops tifffile scikit-image easyocr rapidfuzz pytorch-msssim

from datetime import datetime
import numpy as np
import pandas as pd
import cv2, tifffile
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from rapidfuzz.distance import Levenshtein
from pytorch_msssim import ssim as ssim_torch

assert torch.cuda.is_available(), 'No GPU - Runtime > Change runtime type > GPU'
DEVICE = 'cuda'
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

GPU_NAME = torch.cuda.get_device_name(0)
print('PyTorch:', torch.__version__)
print('GPU    :', GPU_NAME)
print('VRAM   :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
if 'T4' in GPU_NAME:
    print('NOTE: T4 works but full training takes several hours (resumable). A100 / L4 is much faster.')

LOG_TXT = f'{ULT_LOG}/train_log.txt'

def log(msg):
    print(msg)
    with open(LOG_TXT, 'a') as fp:
        fp.write(f'[{datetime.now():%Y-%m-%d %H:%M:%S}] {msg}\n')

## 1. Preprocessing helpers (loading, splitting, registration)

In [ ]:
MARGIN = 48                                  # pixels trimmed from every border after warping
SIGMA_CANDIDATES = list(range(5, 105, 5))
N_TEST_PER_FONT = 20
N_VAL_PER_FONT = 5

def sample_re(level):
    return re.compile(r'^focusStep_' + str(level) + r'_(timesR|verdanaRef)_size_\d+_sample_(\d{4})\.tif$')

def read_raw(path):
    img = tifffile.imread(path).astype(np.float32)
    if img.ndim == 3:
        img = img.mean(axis=-1)
    return img

def normalise(img):
    bg = np.percentile(img, 99.5)
    if bg <= 0:
        bg = img.max() + 1e-6
    return np.clip(img / bg, 0.0, 1.0)

def load_pair(p):
    # HDC 2021: CAM01 = focused (ground truth), CAM02 = defocused (input)
    s = normalise(read_raw(p['cam01']))
    b = normalise(read_raw(p['cam02']))
    assert s.shape == b.shape, p['id']
    return b, s

def scan_pairs(root_dir, level):
    rx = sample_re(level)
    pairs = []
    for root, dirs, files in os.walk(root_dir):
        if '__MACOSX' in root or os.path.basename(root) != 'CAM01':
            continue
        cam2_dir = os.path.join(os.path.dirname(root), 'CAM02')
        if not os.path.isdir(cam2_dir):
            continue
        cam1_set = set(f for f in files if rx.match(f))
        cam2_set = set(f for f in os.listdir(cam2_dir) if rx.match(f))
        for f in sorted(cam1_set & cam2_set):
            m = rx.match(f)
            pairs.append({'id': f'{m.group(1)}_{m.group(2)}', 'font': m.group(1),
                          'cam01': os.path.join(root, f), 'cam02': os.path.join(cam2_dir, f),
                          'txt': os.path.join(root, f.replace('.tif', '.txt'))})
    pairs.sort(key=lambda p: p['id'])
    return pairs

def edge_energy(img):
    sm = cv2.GaussianBlur(img, (0, 0), 2.0)
    gy, gx = np.gradient(sm)
    return float(np.mean(gx ** 2 + gy ** 2))

def make_split(pairs):
    rng = random.Random(SEED)
    split = {'train': [], 'val': [], 'test': []}
    for font in ['timesR', 'verdanaRef']:
        ids = sorted(p['id'] for p in pairs if p['font'] == font)
        assert len(ids) > N_TEST_PER_FONT + N_VAL_PER_FONT, f'Too few {font} images: {len(ids)}'
        rng.shuffle(ids)
        split['test'] += ids[:N_TEST_PER_FONT]
        split['val'] += ids[N_TEST_PER_FONT:N_TEST_PER_FONT + N_VAL_PER_FONT]
        split['train'] += ids[N_TEST_PER_FONT + N_VAL_PER_FONT:]
    return split

def blur_for_matching(img, sigma):
    return cv2.GaussianBlur(img, (0, 0), sigma)

def estimate_blur_sigma(s, b):
    H, W = s.shape
    cy, cx = H // 2, W // 2
    R = 60
    region = b[cy - 300 - R:cy + 300 + R, cx - 800 - R:cx + 800 + R]
    best_sig, best_ncc = 0, -1.0
    for sig in SIGMA_CANDIDATES:
        pad = min(int(3 * sig) + 1, cy - 300, cx - 800)
        sub = s[cy - 300 - pad:cy + 300 + pad, cx - 800 - pad:cx + 800 + pad]
        tmpl = blur_for_matching(sub, sig)[pad:pad + 600, pad:pad + 1600]
        ncc = cv2.matchTemplate(region, tmpl, cv2.TM_CCOEFF_NORMED).max()
        if ncc > best_ncc:
            best_sig, best_ncc = sig, ncc
    return best_sig

def match_points(s, b, sigma, min_ncc=0.8, ph=160, pw=320, R=60, step=80):
    sb = blur_for_matching(s, sigma)
    H, W = s.shape
    src, dst = [], []
    for y in range(R + ph // 2, H - R - ph // 2, step):
        for x in range(R + pw // 2, W - R - pw // 2, step):
            if s[y - ph // 2:y + ph // 2, x - pw // 2:x + pw // 2].std() < 0.08:
                continue
            tmpl = sb[y - ph // 2:y + ph // 2, x - pw // 2:x + pw // 2]
            region = b[y - ph // 2 - R:y + ph // 2 + R, x - pw // 2 - R:x + pw // 2 + R]
            r = cv2.matchTemplate(region, tmpl, cv2.TM_CCOEFF_NORMED)
            _, mx, _, loc = cv2.minMaxLoc(r)
            if mx < min_ncc:
                continue
            src.append((x, y))
            dst.append((x + loc[0] - R, y + loc[1] - R))
    return np.float32(src).reshape(-1, 2), np.float32(dst).reshape(-1, 2)

def warp_gt(s, M):
    # out(x) = s(M x): M maps INPUT pixel coords -> GT pixel coords
    H, W = s.shape
    return cv2.warpAffine(s, np.asarray(M, dtype=np.float64), (W, H),
                          flags=cv2.INTER_CUBIC | cv2.WARP_INVERSE_MAP, borderMode=cv2.BORDER_REPLICATE)

def align_score(s_warped, b, sigma):
    sb = blur_for_matching(s_warped, sigma)
    m = cv2.dilate((cv2.GaussianBlur(b, (0, 0), 5) < 0.9).astype(np.uint8), np.ones((61, 61), np.uint8)) > 0
    m[:80] = False
    m[-80:] = False
    m[:, :80] = False
    m[:, -80:] = False
    return float(np.corrcoef(sb[m], b[m])[0, 1])

def refine_ecc(s, b, M, sigma, scale=0.5):
    """Per-image affine refinement (fixes residual lens / position differences). Same convention as warp_gt."""
    sb = cv2.resize(blur_for_matching(s, sigma), None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    bs = cv2.resize(b, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    Ms = np.asarray(M, dtype=np.float32).copy()
    Ms[:, 2] *= scale
    crit = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 100, 1e-6)
    try:
        _, Ms = cv2.findTransformECC(bs.astype(np.float32), sb.astype(np.float32), Ms,
                                     cv2.MOTION_AFFINE, crit, None, 5)
    except cv2.error:
        return None
    Mr = Ms.astype(np.float64)
    Mr[:, 2] /= scale
    M0 = np.asarray(M, dtype=np.float64)
    if np.abs(Mr[:, :2] - M0[:, :2]).max() > 0.01 or np.abs(Mr[:, 2] - M0[:, 2]).max() > 15:
        return None    # implausibly large jump -> reject
    return Mr

print('Helpers ready')

## 2. Prepare all levels
Each level is copied, unzipped, registered and stored as a compact array on the local disk, then the raw files are deleted.
Registration matrices are cached on Drive, so after a disconnect this step only re-copies and re-warps (no re-matching).

In [ ]:
def fit_global_affine(level, pairs_by_id, split):
    fit_ids = random.Random(SEED).sample(split['train'], min(12, len(split['train'])))
    imgs = {i: load_pair(pairs_by_id[i]) for i in fit_ids}
    sigma = int(np.median([estimate_blur_sigma(imgs[i][1], imgs[i][0]) for i in fit_ids[:4]]))
    if sigma >= SIGMA_CANDIDATES[-1]:
        log(f'  WARNING level {level}: blur sigma hit the search limit ({sigma}px)')
    M = None
    for min_ncc in [0.8, 0.7, 0.6]:
        all_src, all_dst = [], []
        for i in fit_ids:
            src, dst = match_points(imgs[i][1], imgs[i][0], sigma, min_ncc=min_ncc)
            all_src.append(src)
            all_dst.append(dst)
        all_src = np.concatenate(all_src)
        all_dst = np.concatenate(all_dst)
        if len(all_src) >= 50:
            M, inl = cv2.estimateAffine2D(all_dst, all_src, ransacReprojThreshold=3.0, maxIters=5000)
            if M is not None:
                log(f'  level {level}: sigma ~{sigma}px | {len(all_src)} matches | {int(inl.sum())} inliers (NCC>={min_ncc})')
                break
    if M is None:
        prev = f'{ULT_CACHE}/level{level - 1}_registration.json'
        assert os.path.exists(prev), f'Registration failed for level {level} and no fallback'
        M = np.array(json.load(open(prev))['global'])
        log(f'  WARNING level {level}: registration failed, using level {level - 1} affine')
    return sigma, M


def prepare_level(level):
    ldir = f'{LOCAL}/L{level}'
    meta_path = f'{ldir}/meta.json'
    if os.path.exists(meta_path):
        meta = json.load(open(meta_path))
        print(f'Level {level}: already prepared locally')
    else:
        os.makedirs(ldir, exist_ok=True)
        tmp = f'/content/tmp_L{level}'
        shutil.rmtree(tmp, ignore_errors=True)
        os.makedirs(tmp, exist_ok=True)
        zip_src = find_zip(level)
        print(f'Level {level}: copying {os.path.basename(zip_src)} ...')
        shutil.copy(zip_src, f'{tmp}.zip')
        subprocess.run(['unzip', '-q', '-o', f'{tmp}.zip', '-d', tmp], check=True)
        os.remove(f'{tmp}.zip')

        pairs = scan_pairs(tmp, level)
        assert len(pairs) > 0, f'No focusStep_{level}_*.tif pairs found in {zip_src}'
        pairs_by_id = {p['id']: p for p in pairs}

        # sanity: GT camera must be sharper
        es, eb = [], []
        for p in random.Random(SEED).sample(pairs, min(6, len(pairs))):
            b, s = load_pair(p)
            es.append(edge_energy(s))
            eb.append(edge_energy(b))
        assert np.mean(es) > np.mean(eb), f'Level {level}: CAM01 not sharper than CAM02 - check data'

        # split: reuse the per-level split if it exists (keeps test sets identical to the per-level runs)
        old_split = level_result_file(level, 'split.json')
        if old_split is not None:
            split = json.load(open(old_split))
            print(f'  reusing split from {old_split}')
        else:
            split = make_split(pairs)
        tr, va, te = set(split['train']), set(split['val']), set(split['test'])
        assert len(tr & va) == 0 and len(tr & te) == 0 and len(va & te) == 0, 'LEAKAGE'
        assert (tr | va | te) <= set(pairs_by_id), 'split refers to missing images'
        json.dump(split, open(f'{ULT_RES}/splits/level{level}_split.json', 'w'), indent=2)

        # registration (cached on Drive)
        reg_path = f'{ULT_CACHE}/level{level}_registration.json'
        if os.path.exists(reg_path):
            reg = json.load(open(reg_path))
            print('  using cached registration')
        else:
            sigma, Mg = fit_global_affine(level, pairs_by_id, split)
            reg = {'sigma': sigma, 'global': np.asarray(Mg).tolist(), 'per_image': {}, 'checks': []}
            n_ref = 0
            for p in tqdm(pairs, desc=f'L{level} ECC refine'):
                b, s = load_pair(p)
                ncc_raw = align_score(s, b, sigma)
                ncc_glob = align_score(warp_gt(s, Mg), b, sigma)
                Mi = np.asarray(Mg)
                ncc_final = ncc_glob
                Mr = refine_ecc(s, b, Mg, sigma)
                if Mr is not None:
                    ncc_ref = align_score(warp_gt(s, Mr), b, sigma)
                    if ncc_ref > ncc_glob:
                        Mi = Mr
                        ncc_final = ncc_ref
                        n_ref += 1
                reg['per_image'][p['id']] = Mi.tolist()
                reg['checks'].append({'id': p['id'], 'ncc_before': ncc_raw, 'ncc_global': ncc_glob, 'ncc_final': ncc_final})
            reg['n_refined'] = n_ref
            json.dump(reg, open(reg_path, 'w'))

        chk = pd.DataFrame(reg['checks'])
        chk.to_csv(f'{ULT_RES}/registration/level{level}_pairing_checks.csv', index=False)

        # warp + trim + write to local arrays
        b0, s0 = load_pair(pairs[0])
        H, W = b0.shape[0] - 2 * MARGIN, b0.shape[1] - 2 * MARGIN
        N = len(pairs)
        blur_mm = np.lib.format.open_memmap(f'{ldir}/blur.npy', mode='w+', dtype=np.float16, shape=(N, H, W))
        gt_mm = np.lib.format.open_memmap(f'{ldir}/gt.npy', mode='w+', dtype=np.float16, shape=(N, H, W))
        texts = {}
        for i, p in enumerate(tqdm(pairs, desc=f'L{level} warp')):
            b, s = load_pair(p)
            w = np.clip(warp_gt(s, reg['per_image'][p['id']]), 0, 1)
            blur_mm[i] = b[MARGIN:-MARGIN, MARGIN:-MARGIN].astype(np.float16)
            gt_mm[i] = w[MARGIN:-MARGIN, MARGIN:-MARGIN].astype(np.float16)
            if os.path.exists(p['txt']):
                texts[p['id']] = open(p['txt'], encoding='utf-8', errors='ignore').read()
            else:
                texts[p['id']] = ''
        blur_mm.flush()
        gt_mm.flush()
        del blur_mm, gt_mm

        meta = {'level': level, 'ids': [p['id'] for p in pairs], 'fonts': [p['font'] for p in pairs],
                'texts': texts, 'split': split, 'sigma': reg['sigma'], 'shape': [N, H, W],
                'ncc_before': float(chk.ncc_before.mean()), 'ncc_global': float(chk.ncc_global.mean()),
                'ncc_final': float(chk.ncc_final.mean()), 'n_refined': int(reg.get('n_refined', 0))}
        json.dump(meta, open(meta_path, 'w'))
        shutil.rmtree(tmp, ignore_errors=True)

    d = dict(meta)
    d['blur'] = np.load(f'{ldir}/blur.npy', mmap_mode='r')
    d['gt'] = np.load(f'{ldir}/gt.npy', mmap_mode='r')
    d['index'] = {pid: i for i, pid in enumerate(meta['ids'])}
    d['font_of'] = {pid: f for pid, f in zip(meta['ids'], meta['fonts'])}
    return d


DATA = {}
for lv in LEVELS:
    DATA[lv] = prepare_level(lv)

rows = []
for lv in LEVELS:
    d = DATA[lv]
    rows.append({'level': lv, 'pairs': d['shape'][0], 'blur_sigma_px': d['sigma'],
                 'ncc_before': round(d['ncc_before'], 3), 'ncc_global_affine': round(d['ncc_global'], 3),
                 'ncc_after_ecc': round(d['ncc_final'], 3), 'images_refined': d['n_refined'],
                 'train': len(d['split']['train']), 'val': len(d['split']['val']), 'test': len(d['split']['test'])})
prep = pd.DataFrame(rows)
prep.to_csv(f'{ULT_RES}/registration/preprocessing_summary.csv', index=False)
print(prep.to_string(index=False))
total, used, free = shutil.disk_usage('/content')
print(f'Local disk free: {free / 1e9:.0f} GB')

In [ ]:
# Visual check: one input / GT crop per level (input must look blurred, GT sharp and aligned)
fig, axes = plt.subplots(len(LEVELS), 2, figsize=(14, 2.2 * len(LEVELS)))
for r, lv in enumerate(LEVELS):
    d = DATA[lv]
    i = d['index'][d['split']['test'][0]]
    H, W = d['shape'][1], d['shape'][2]
    y0, x0 = H // 2 - 150, W // 2 - 500
    axes[r, 0].imshow(np.asarray(d['blur'][i, y0:y0 + 300, x0:x0 + 1000], dtype=np.float32), cmap='gray', vmin=0, vmax=1)
    axes[r, 1].imshow(np.asarray(d['gt'][i, y0:y0 + 300, x0:x0 + 1000], dtype=np.float32), cmap='gray', vmin=0, vmax=1)
    axes[r, 0].set_ylabel(f'L{lv}  ~{d["sigma"]}px')
    for a in axes[r]:
        a.set_xticks([])
        a.set_yticks([])
axes[0, 0].set_title('Input (CAM02)')
axes[0, 1].set_title('Registered GT (CAM01)')
plt.tight_layout()
plt.savefig(f'{ULT_FIG}/data_check.png', dpi=110)
plt.show()

## 3. Model

In [ ]:
import importlib.util
from torch.utils.checkpoint import checkpoint

RESTORMER_DIR = '/content/Restormer'
if not os.path.exists(RESTORMER_DIR):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/swz30/Restormer.git', RESTORMER_DIR], check=True)

spec = importlib.util.spec_from_file_location('restormer_arch', f'{RESTORMER_DIR}/basicsr/models/archs/restormer_arch.py')
arch = importlib.util.module_from_spec(spec)
spec.loader.exec_module(arch)

def load_weights(path):
    ck = torch.load(path, map_location='cpu', weights_only=False)
    if isinstance(ck, dict) and 'params' in ck:
        ck = ck['params']
    if isinstance(ck, dict) and 'model' in ck and 'opt' in ck:
        ck = ck['model']
    return ck

def build_model(weights=PRETRAINED):
    m = arch.Restormer(inp_channels=3, out_channels=3, dim=48, num_blocks=[4, 6, 6, 8],
                       num_refinement_blocks=4, heads=[1, 2, 4, 8], ffn_expansion_factor=2.66,
                       bias=False, LayerNorm_type='WithBias', dual_pixel_task=False)
    m.load_state_dict(load_weights(weights), strict=True)
    return m.to(DEVICE)

def enable_grad_checkpointing(m):
    """Recompute each transformer block in the backward pass -> large patches fit in GPU memory.
    Parameter names are unchanged, so checkpoints stay compatible."""
    n = 0
    for mod in m.modules():
        if isinstance(mod, arch.TransformerBlock):
            f = mod.forward
            def ck_forward(x, _f=f):
                if torch.is_grad_enabled():
                    return checkpoint(_f, x, use_reentrant=False)
                return _f(x)
            mod.forward = ck_forward
            n += 1
    return n

_m = build_model()
print('Total params:', round(sum(p.numel() for p in _m.parameters()) / 1e6, 2), 'M')
del _m
torch.cuda.empty_cache()

In [ ]:
USE_AMP = True
TILE, OVERLAP = 768, 256

@torch.no_grad()
def restore(model, img2d, tile=TILE, overlap=OVERLAP):
    model.eval()
    H, W = img2d.shape
    ph = max(tile, int(np.ceil(H / 8) * 8)) - H
    pw = max(tile, int(np.ceil(W / 8) * 8)) - W
    x = torch.from_numpy(np.ascontiguousarray(img2d, dtype=np.float32))[None, None].repeat(1, 3, 1, 1).to(DEVICE)
    x = F.pad(x, (0, pw, 0, ph), mode='reflect')
    Hp, Wp = x.shape[-2:]
    stride = tile - overlap

    def starts(n):
        if n <= tile:
            return [0]
        s = list(range(0, n - tile, stride))
        s.append(n - tile)
        return s

    ramp = torch.ones(tile, device=DEVICE)
    edge = torch.linspace(0.05, 1.0, overlap // 2, device=DEVICE)
    ramp[:overlap // 2] = edge
    ramp[-(overlap // 2):] = edge.flip(0)
    win = (ramp[:, None] * ramp[None, :])[None, None]

    out = torch.zeros_like(x)
    wgt = torch.zeros((1, 1, Hp, Wp), device=DEVICE)
    for y in starts(Hp):
        for x0 in starts(Wp):
            patch = x[:, :, y:y + tile, x0:x0 + tile]
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
                r = model(patch)
            out[:, :, y:y + tile, x0:x0 + tile] += r.float() * win
            wgt[:, :, y:y + tile, x0:x0 + tile] += win
    res = (out / wgt)[:, :, :H, :W].mean(1)[0].clamp(0, 1)
    return res.cpu().numpy()

FLIPS = [(False, False), (True, False), (False, True), (True, True)]

def restore_tta(model, img2d):
    acc = np.zeros(img2d.shape, dtype=np.float32)
    for fh, fv in FLIPS:
        x = img2d
        if fh:
            x = x[:, ::-1]
        if fv:
            x = x[::-1, :]
        y = restore(model, x)
        if fv:
            y = y[::-1, :]
        if fh:
            y = y[:, ::-1]
        acc += y
    return acc / len(FLIPS)

def img_metrics(pred, gt):
    pred = pred.astype(np.float32)
    gt = gt.astype(np.float32)
    psnr = peak_signal_noise_ratio(gt, pred, data_range=1.0)
    ssim = structural_similarity(gt, pred, data_range=1.0)
    # PSNR on the text area only (blank paper inflates the full-image number)
    mask = cv2.dilate((gt < 0.5).astype(np.uint8), np.ones((15, 15), np.uint8)) > 0
    if mask.sum() > 0:
        mse = float(np.mean((pred[mask] - gt[mask]) ** 2))
        psnr_text = 10 * np.log10(1.0 / max(mse, 1e-10))
    else:
        psnr_text = float('nan')
    return psnr, ssim, psnr_text

def save_png16(img, path):
    cv2.imwrite(path, (np.clip(img.astype(np.float32), 0, 1) * 65535).round().astype(np.uint16))

print('Inference helpers ready')

## 4. Training setup

In [ ]:
# Progressive training: (iterations up to, patch size, batch). Effective batch is kept by gradient accumulation
# if the GPU cannot fit the requested batch.
STAGES = [
    (2500, 160, 4),
    (5000, 224, 2),
    (6000, 288, 2),
]
TOTAL_ITERS  = STAGES[-1][0]
LR           = 8e-5        # decoder / refinement
ENC_LR_MULT  = 0.5         # encoder learns slower (keeps pretrained features)
WARMUP       = 500
MIN_LR_FRAC  = 0.01
W_FFT        = 0.05
W_SSIM       = 0.1
EMA_DECAY    = 0.999
VAL_EVERY    = 1000
SAVE_EVERY   = 500
TEXT_STD     = 0.05
LEVEL_WEIGHT = {lv: 1.0 + 0.15 * (lv - 5) for lv in LEVELS}   # sample harder levels a bit more

LAST_PATH = f'{ULT_CKPT}/ultimate_last.pth'
BEST_PATH = f'{ULT_CKPT}/ultimate_best.pth'      # EMA weights, best mean val SSIM

ENC_PREFIX = ('patch_embed', 'encoder_level1', 'down1_2', 'encoder_level2', 'down2_3', 'encoder_level3', 'down3_4')

model = build_model(PRETRAINED)
n_ck = enable_grad_checkpointing(model)
enc_params, dec_params = [], []
for name, p in model.named_parameters():
    p.requires_grad = True
    if name.startswith(ENC_PREFIX):
        enc_params.append(p)
    else:
        dec_params.append(p)
all_params = enc_params + dec_params
print(f'Gradient checkpointing on {n_ck} blocks | encoder {sum(p.numel() for p in enc_params) / 1e6:.2f}M '
      f'| decoder {sum(p.numel() for p in dec_params) / 1e6:.2f}M (all trainable)')

optimizer = torch.optim.AdamW([{'params': enc_params, 'lr': LR * ENC_LR_MULT},
                               {'params': dec_params, 'lr': LR}],
                              weight_decay=1e-4, betas=(0.9, 0.999))

def lr_lambda(step):
    if step < WARMUP:
        return (step + 1) / WARMUP
    t = (step - WARMUP) / max(1, TOTAL_ITERS - WARMUP)
    return MIN_LR_FRAC + (1 - MIN_LR_FRAC) * 0.5 * (1 + math.cos(math.pi * min(1.0, t)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps = eps
    def forward(self, x, y):
        return torch.mean(torch.sqrt((x - y) ** 2 + self.eps ** 2))

charbonnier = CharbonnierLoss()

def fft_loss(x, y):
    return torch.mean(torch.abs(torch.fft.rfft2(x, norm='ortho') - torch.fft.rfft2(y, norm='ortho')))

def total_loss(out, gt):
    out = out.float()
    l_pix = charbonnier(out, gt)
    l_fft = fft_loss(out, gt)
    og = out.mean(1, keepdim=True).clamp(0, 1)
    gg = gt.mean(1, keepdim=True)
    l_ssim = 1 - ssim_torch(og, gg, data_range=1.0, size_average=True)
    return l_pix + W_FFT * l_fft + W_SSIM * l_ssim, (l_pix.item(), l_fft.item(), l_ssim.item())

# ---- EMA ----
ema = {k: v.detach().clone().float() for k, v in model.state_dict().items()}

@torch.no_grad()
def ema_update(step):
    d = min(EMA_DECAY, (1 + step) / (10 + step))
    msd = model.state_dict()
    for k, v in ema.items():
        if v.dtype.is_floating_point:
            v.mul_(d).add_(msd[k].detach().float(), alpha=1 - d)

eval_model = build_model(PRETRAINED)
eval_model.eval()

# ---- data sampling ----
level_list = list(LEVELS)
level_w = np.array([LEVEL_WEIGHT[lv] for lv in level_list], dtype=np.float64)
level_w = level_w / level_w.sum()

def random_patch(patch):
    lv = level_list[np.random.choice(len(level_list), p=level_w)]
    d = DATA[lv]
    N, H, W = d['shape']
    for attempt in range(20):
        pid = random.choice(d['split']['train'])
        i = d['index'][pid]
        y = random.randint(0, H - patch)
        x = random.randint(0, W - patch)
        g = np.asarray(d['gt'][i, y:y + patch, x:x + patch], dtype=np.float32)
        if g.std() > TEXT_STD:
            break
    b = np.asarray(d['blur'][i, y:y + patch, x:x + patch], dtype=np.float32)
    k = random.randint(0, 3)
    b = np.rot90(b, k)
    g = np.rot90(g, k)
    if random.random() < 0.5:
        b = np.fliplr(b)
        g = np.fliplr(g)
    return b.copy(), g.copy()

def make_batch(n, patch):
    bs, gs = [], []
    for _ in range(n):
        b, g = random_patch(patch)
        bs.append(b)
        gs.append(g)
    b = torch.from_numpy(np.stack(bs))[:, None].repeat(1, 3, 1, 1).to(DEVICE, non_blocking=True)
    g = torch.from_numpy(np.stack(gs))[:, None].repeat(1, 3, 1, 1).to(DEVICE, non_blocking=True)
    return b, g

def probe_micro_batch(patch, batch):
    """Largest micro-batch (<= batch) that fits in memory for this patch size."""
    mb = batch
    while mb >= 1:
        try:
            x = torch.rand(mb, 3, patch, patch, device=DEVICE)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
                y = model(x)
            loss, _ = total_loss(y, x)
            loss.backward()
            model.zero_grad(set_to_none=True)
            del x, y, loss
            torch.cuda.empty_cache()
            return mb
        except torch.cuda.OutOfMemoryError:
            model.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
            mb = mb // 2
    raise RuntimeError(f'Patch {patch} does not fit even with batch 1 - reduce STAGES patch sizes')

def stage_for(step):
    for until, patch, batch in STAGES:
        if step < until:
            return until, patch, batch
    return STAGES[-1]

@torch.no_grad()
def validate():
    eval_model.load_state_dict(ema)
    per_level = {}
    for lv in LEVELS:
        d = DATA[lv]
        ps, ss = [], []
        for vid in d['split']['val']:
            i = d['index'][vid]
            blur = np.asarray(d['blur'][i], dtype=np.float32)
            sharp = np.asarray(d['gt'][i], dtype=np.float32)
            p, s, _ = img_metrics(restore(eval_model, blur), sharp)
            ps.append(p)
            ss.append(s)
        per_level[lv] = (float(np.mean(ps)), float(np.mean(ss)))
    mean_p = float(np.mean([v[0] for v in per_level.values()]))
    mean_s = float(np.mean([v[1] for v in per_level.values()]))
    return mean_p, mean_s, per_level

print(f'Setup ready | {TOTAL_ITERS} iterations | stages {STAGES}')

## 5. Train (resumable)

In [ ]:
start_iter, best_ssim = 0, -1.0
hist = {'iter': [], 'loss': [], 'l_pix': [], 'l_fft': [], 'l_ssim': [], 'lr': [],
        'val_iter': [], 'val_psnr': [], 'val_ssim': [], 'val_per_level': []}

if RESUME and os.path.exists(LAST_PATH):
    st = torch.load(LAST_PATH, map_location='cpu', weights_only=False)
    model.load_state_dict(st['model'])
    optimizer.load_state_dict(st['opt'])
    scheduler.load_state_dict(st['sched'])
    scaler.load_state_dict(st['scaler'])
    for k in ema:
        ema[k].copy_(st['ema'][k].to(DEVICE))
    start_iter, best_ssim, hist = st['iter'], st['best_ssim'], st['hist']
    random.setstate(st['py_rng'])
    np.random.set_state(st['np_rng'])
    torch.set_rng_state(st['torch_rng'])
    log(f'Resumed from iter {start_iter} (best mean val SSIM {best_ssim:.4f})')
else:
    log(f'New ultimate run | levels {LEVELS} | stages {STAGES} | LR {LR} (enc x{ENC_LR_MULT}) | GPU {GPU_NAME}')
    vp, vs, per = validate()
    hist['val_iter'].append(0)
    hist['val_psnr'].append(vp)
    hist['val_ssim'].append(vs)
    hist['val_per_level'].append({str(k): v for k, v in per.items()})
    log(f'Iter 0 (pretrained) val: PSNR {vp:.2f} | SSIM {vs:.4f} | ' +
        ' '.join(f'L{k}:{v[1]:.3f}' for k, v in per.items()))

def save_last(step):
    torch.save({'model': model.state_dict(), 'opt': optimizer.state_dict(), 'sched': scheduler.state_dict(),
                'scaler': scaler.state_dict(), 'ema': {k: v.cpu() for k, v in ema.items()},
                'iter': step, 'best_ssim': best_ssim, 'hist': hist,
                'py_rng': random.getstate(), 'np_rng': np.random.get_state(), 'torch_rng': torch.get_rng_state()},
               LAST_PATH + '.tmp')
    os.replace(LAST_PATH + '.tmp', LAST_PATH)

model.train()
cur_patch, micro, accum = None, None, None
run_loss, run_parts, run_n = 0.0, np.zeros(3), 0
pbar = tqdm(range(start_iter, TOTAL_ITERS), initial=start_iter, total=TOTAL_ITERS)
for it in pbar:
    until, patch, batch = stage_for(it)
    if patch != cur_patch:
        micro = probe_micro_batch(patch, batch)
        accum = int(math.ceil(batch / micro))
        cur_patch = patch
        model.train()
        log(f'Stage: patch {patch}px | batch {batch} = micro {micro} x accum {accum} | until iter {until}')

    optimizer.zero_grad(set_to_none=True)
    parts_acc = np.zeros(3)
    loss_acc = 0.0
    for a in range(accum):
        b, g = make_batch(micro, patch)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
            out = model(b)
        loss, parts = total_loss(out, g)
        scaler.scale(loss / accum).backward()
        loss_acc += loss.item() / accum
        parts_acc += np.array(parts) / accum
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(all_params, 1.0)
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()
    ema_update(it)

    run_loss += loss_acc
    run_parts += parts_acc
    run_n += 1
    step = it + 1
    if step % 50 == 0:
        hist['iter'].append(step)
        hist['loss'].append(run_loss / run_n)
        hist['l_pix'].append(run_parts[0] / run_n)
        hist['l_fft'].append(run_parts[1] / run_n)
        hist['l_ssim'].append(run_parts[2] / run_n)
        hist['lr'].append(scheduler.get_last_lr()[1])
        pbar.set_postfix(loss=f'{run_loss / run_n:.4f}', patch=patch, lr=f'{scheduler.get_last_lr()[1]:.2e}')
        run_loss, run_parts, run_n = 0.0, np.zeros(3), 0

    if step % VAL_EVERY == 0 or step == TOTAL_ITERS:
        vp, vs, per = validate()
        model.train()
        hist['val_iter'].append(step)
        hist['val_psnr'].append(vp)
        hist['val_ssim'].append(vs)
        hist['val_per_level'].append({str(k): v for k, v in per.items()})
        tag = ''
        if vs > best_ssim:
            best_ssim = vs
            torch.save({k: v.cpu() for k, v in ema.items()}, BEST_PATH)
            tag = '  <- best, saved'
        log(f'Iter {step}: val PSNR {vp:.2f} | SSIM {vs:.4f}{tag} | ' +
            ' '.join(f'L{k}:{v[0]:.2f}/{v[1]:.3f}' for k, v in per.items()))
        json.dump(hist, open(f'{ULT_LOG}/training_history.json', 'w'))

    if step % SAVE_EVERY == 0 or step == TOTAL_ITERS:
        save_last(step)

json.dump(hist, open(f'{ULT_LOG}/training_history.json', 'w'))
log(f'Training done. Best mean val SSIM: {best_ssim:.4f}')

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(22, 4))
ax[0].plot(hist['iter'], hist['loss'], label='total')
ax[0].plot(hist['iter'], hist['l_pix'], label='charbonnier', alpha=0.7)
ax[0].set_title('Training loss')
ax[0].legend()
ax[1].plot(hist['val_iter'], hist['val_psnr'], marker='o')
ax[1].set_title('Mean val PSNR (dB)')
ax[2].plot(hist['val_iter'], hist['val_ssim'], marker='o', color='green')
ax[2].set_title('Mean val SSIM')
for lv in LEVELS:
    ys = [v[str(lv)][1] for v in hist['val_per_level']]
    ax[3].plot(hist['val_iter'], ys, marker='.', label=f'L{lv}')
ax[3].set_title('Val SSIM per level')
ax[3].legend(fontsize=8)
for a in ax:
    a.set_xlabel('iteration')
for until, _, _ in STAGES[:-1]:
    for a in ax:
        a.axvline(until, color='grey', ls=':', lw=0.8)
plt.tight_layout()
plt.savefig(f'{ULT_FIG}/training_curves.png', dpi=150)
plt.show()

## 6. Evaluate on every level's test set
Methods: blurred input, original pretrained Restormer, per-level fine-tuned model (where one exists), ultimate model, ultimate + TTA, and OCR on sharp GT as the upper bound.
Each level is saved as soon as it finishes, so a disconnect only repeats the unfinished level.

In [ ]:
import easyocr
reader = easyocr.Reader(['en'], gpu=True)

def to_u8(img):
    return (np.clip(img.astype(np.float32), 0, 1) * 255).round().astype(np.uint8)

def ocr_text(img):
    out = reader.readtext(to_u8(img), detail=1, paragraph=False)
    if len(out) == 0:
        return ''
    items = []
    for box, txt, conf in out:
        ys = [pt[1] for pt in box]
        xs = [pt[0] for pt in box]
        items.append((np.mean(ys), min(xs), max(ys) - min(ys), txt))
    items.sort(key=lambda t: t[0])
    lines = []
    for itm in items:
        if len(lines) > 0 and abs(itm[0] - lines[-1][0][0]) < 0.5 * itm[2]:
            lines[-1].append(itm)
        else:
            lines.append([itm])
    text_lines = []
    for ln in lines:
        ln.sort(key=lambda t: t[1])
        text_lines.append(' '.join(t[3] for t in ln))
    return ' '.join(text_lines)

def clean(s):
    return ' '.join(s.split())

def char_acc(pred, gt):
    gt_c = clean(gt)
    if len(gt_c) == 0:
        return float('nan')
    d = Levenshtein.distance(clean(pred), gt_c)
    return max(0.0, 1.0 - d / len(gt_c))

def token_acc(pred, gt):
    gt_tok = clean(gt).split()
    if len(gt_tok) == 0:
        return float('nan')
    pred_tok = clean(pred).split()
    hits = 0
    for t in gt_tok:
        if t in pred_tok:
            pred_tok.remove(t)
            hits += 1
    return hits / len(gt_tok)

model_pre = build_model(PRETRAINED).eval()
model_ult = build_model(PRETRAINED)
model_ult.load_state_dict(torch.load(BEST_PATH, map_location='cpu'))
model_ult.eval()
for _name in ['model', 'eval_model', 'optimizer']:
    if _name in globals():
        del globals()[_name]
torch.cuda.empty_cache()

M_BLUR, M_PRE, M_LVL, M_ULT, M_TTA, M_GT = ('Blurred input', 'Pretrained Restormer', 'Per-level fine-tuned',
                                             'Ultimate (all levels)', 'Ultimate + TTA', 'Sharp GT (upper bound)')
CROP_H, CROP_W = 400, 1200

for lv in LEVELS:
    out_csv = f'{ULT_RES}/per_level/level{lv}_per_image.csv'
    if os.path.exists(out_csv):
        print(f'Level {lv}: already evaluated')
        continue
    d = DATA[lv]
    lvl_path = level_ckpt(lv, 'best')
    model_lvl = None
    if lvl_path is not None:
        model_lvl = build_model(PRETRAINED)
        model_lvl.load_state_dict(load_weights(lvl_path))
        model_lvl.eval()

    if SAVE_RESTORED:
        os.makedirs(f'{ULT_REST}/level{lv}', exist_ok=True)

    rows, ocr_rows, crops = [], [], {}
    for n, tid in enumerate(tqdm(d['split']['test'], desc=f'Level {lv}')):
        i = d['index'][tid]
        blur = np.asarray(d['blur'][i], dtype=np.float32)
        sharp = np.asarray(d['gt'][i], dtype=np.float32)
        outs = {M_BLUR: blur, M_PRE: restore(model_pre, blur)}
        if model_lvl is not None:
            outs[M_LVL] = restore(model_lvl, blur)
        outs[M_ULT] = restore(model_ult, blur)
        if USE_TTA:
            outs[M_TTA] = restore_tta(model_ult, blur)

        for method, img in outs.items():
            p, s, pt = img_metrics(img, sharp)
            pred = ocr_text(img)
            rows.append({'level': lv, 'method': method, 'id': tid, 'font': d['font_of'][tid], 'psnr': p,
                         'ssim': s, 'psnr_text': pt, 'char_acc': char_acc(pred, d['texts'][tid]),
                         'token_acc': token_acc(pred, d['texts'][tid])})
            ocr_rows.append({'level': lv, 'method': method, 'id': tid, 'gt': clean(d['texts'][tid]), 'ocr': clean(pred)})
        pred = ocr_text(sharp)
        rows.append({'level': lv, 'method': M_GT, 'id': tid, 'font': d['font_of'][tid], 'psnr': np.nan,
                     'ssim': np.nan, 'psnr_text': np.nan, 'char_acc': char_acc(pred, d['texts'][tid]),
                     'token_acc': token_acc(pred, d['texts'][tid])})
        ocr_rows.append({'level': lv, 'method': M_GT, 'id': tid, 'gt': clean(d['texts'][tid]), 'ocr': clean(pred)})

        if SAVE_RESTORED:
            save_png16(outs[M_ULT], f'{ULT_REST}/level{lv}/{tid}_ultimate.png')
            if USE_TTA:
                save_png16(outs[M_TTA], f'{ULT_REST}/level{lv}/{tid}_ultimate_tta.png')

        if n == 0:   # keep a central crop of the first test image for the comparison figure
            H, W = sharp.shape
            y0, x0 = H // 2 - CROP_H // 2, W // 2 - CROP_W // 2
            for method, img in outs.items():
                crops[method] = img[y0:y0 + CROP_H, x0:x0 + CROP_W].astype(np.float16)
            crops[M_GT] = sharp[y0:y0 + CROP_H, x0:x0 + CROP_W].astype(np.float16)

    pd.DataFrame(ocr_rows).to_csv(f'{ULT_RES}/per_level/level{lv}_ocr_outputs.csv', index=False)
    np.savez_compressed(f'{ULT_FIG}/crops_level{lv}.npz', **{k.replace(' ', '_'): v for k, v in crops.items()})
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    if model_lvl is not None:
        del model_lvl
        torch.cuda.empty_cache()
    s = pd.DataFrame(rows).groupby('method')[['psnr', 'ssim', 'char_acc']].mean()
    print(f'Level {lv}:\n{s.round(4).to_string()}\n')

## 7. Results tables

In [ ]:
ORDER = [M_BLUR, M_PRE, M_LVL, M_ULT, M_TTA, M_GT]
df = pd.concat([pd.read_csv(f'{ULT_RES}/per_level/level{lv}_per_image.csv') for lv in LEVELS], ignore_index=True)
df.to_csv(f'{ULT_RES}/all_levels_per_image.csv', index=False)
pd.concat([pd.read_csv(f'{ULT_RES}/per_level/level{lv}_ocr_outputs.csv') for lv in LEVELS],
          ignore_index=True).to_csv(f'{ULT_RES}/all_levels_ocr_outputs.csv', index=False)

cols = ['psnr', 'ssim', 'psnr_text', 'char_acc', 'token_acc']
summary = df.groupby(['level', 'method'])[cols].agg(['mean', 'std'])
summary.columns = [f'{a}_{b}' for a, b in summary.columns]
summary = summary.reset_index()
summary['method'] = pd.Categorical(summary['method'], [m for m in ORDER if m in summary['method'].unique()], ordered=True)
summary = summary.sort_values(['level', 'method'])
summary.to_csv(f'{ULT_RES}/summary_by_level.csv', index=False)

# Compact table: mean only
compact = df.groupby(['level', 'method'])[cols].mean().reset_index()
compact['method'] = pd.Categorical(compact['method'], [m for m in ORDER if m in compact['method'].unique()], ordered=True)
compact = compact.sort_values(['level', 'method'])
print('=== Test set, mean per level (n = 40 images per level) ===')
print(compact.round(4).to_string(index=False))

# note: 'Per-level fine-tuned' only exists for levels that were trained separately, so its overall mean covers fewer levels
overall = df.groupby('method')[cols].mean().reindex([m for m in ORDER if m in df['method'].unique()])
overall.to_csv(f'{ULT_RES}/summary_overall.csv')
print('\n=== Mean over all levels ===')
print(overall.round(4).to_string())

by_font = df.groupby(['level', 'method', 'font'])[['psnr', 'ssim', 'char_acc']].mean().reset_index()
by_font.to_csv(f'{ULT_RES}/summary_by_font.csv', index=False)

# Gains of the ultimate model
best_ult = M_ULT
if USE_TTA:
    best_ult = M_TTA
gain_rows = []
for lv in LEVELS:
    c = compact[compact.level == lv].set_index('method')
    row = {'level': lv}
    for ref in [M_BLUR, M_PRE, M_LVL]:
        if ref in c.index:
            row[f'dPSNR_vs_{ref}'] = c.loc[best_ult, 'psnr'] - c.loc[ref, 'psnr']
            row[f'dSSIM_vs_{ref}'] = c.loc[best_ult, 'ssim'] - c.loc[ref, 'ssim']
            row[f'dCharAcc_pts_vs_{ref}'] = 100 * (c.loc[best_ult, 'char_acc'] - c.loc[ref, 'char_acc'])
    gain_rows.append(row)
gains = pd.DataFrame(gain_rows)
gains.to_csv(f'{ULT_RES}/gains.csv', index=False)
print(f'\n=== Gains of "{best_ult}" ===')
print(gains.round(3).to_string(index=False))

In [ ]:
# Trend across blur levels
fig, ax = plt.subplots(1, 4, figsize=(22, 4.2))
for i, metric in enumerate(['psnr', 'ssim', 'psnr_text', 'char_acc']):
    for m in ORDER:
        sub = compact[compact.method == m].sort_values('level')
        if len(sub) > 0 and sub[metric].notna().any():
            ax[i].plot(sub.level, sub[metric], marker='o', label=m)
    ax[i].set_title(metric.upper())
    ax[i].set_xlabel('blur level')
    ax[i].set_xticks(LEVELS)
ax[3].legend(fontsize=8)
plt.tight_layout()
plt.savefig(f'{ULT_FIG}/trend_across_levels.png', dpi=150)
plt.show()

# Visual comparison: one test image per level
fig, axes = plt.subplots(len(LEVELS), len(ORDER), figsize=(4.2 * len(ORDER), 1.6 * len(LEVELS) + 0.6))
for r, lv in enumerate(LEVELS):
    z = np.load(f'{ULT_FIG}/crops_level{lv}.npz')
    tid = DATA[lv]['split']['test'][0]
    for c, m in enumerate(ORDER):
        a = axes[r, c]
        a.set_xticks([])
        a.set_yticks([])
        key = m.replace(' ', '_')
        if key in z.files:
            a.imshow(z[key].astype(np.float32), cmap='gray', vmin=0, vmax=1)
            row = df[(df.level == lv) & (df.method == m) & (df.id == tid)]
            if m != M_GT and len(row) > 0:
                a.set_xlabel(f"PSNR {row.psnr.iloc[0]:.2f} | SSIM {row.ssim.iloc[0]:.3f} | Char {100 * row.char_acc.iloc[0]:.0f}%", fontsize=8)
        else:
            a.axis('off')
        if r == 0:
            a.set_title(m, fontsize=10)
        if c == 0:
            a.set_ylabel(f'Level {lv}', fontsize=10)
plt.tight_layout()
plt.savefig(f'{ULT_FIG}/comparison_all_levels.png', dpi=150)
plt.show()

In [ ]:
manifest = {
    'model': 'Restormer, full fine-tuning on HDC2021 focus steps ' + ', '.join(str(l) for l in LEVELS),
    'updated': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'pretrained_base': '../pretrained/single_image_defocus_deblurring.pth',
    'best_checkpoint': 'checkpoints/ultimate_best.pth (EMA weights, best mean val SSIM)',
    'last_checkpoint': 'checkpoints/ultimate_last.pth (full training state)',
    'train_config': {'stages_until_patch_batch': STAGES, 'lr': LR, 'encoder_lr_mult': ENC_LR_MULT,
                     'warmup': WARMUP, 'loss': f'Charbonnier + {W_FFT}*FFT + {W_SSIM}*(1-SSIM)',
                     'ema_decay': EMA_DECAY, 'level_sampling_weight': LEVEL_WEIGHT, 'tile': TILE, 'overlap': OVERLAP},
    'registration': prep.to_dict(orient='records'),
    'best_mean_val_ssim': round(best_ssim, 4),
    'test_overall': overall.round(4).reset_index().to_dict(orient='records'),
}
json.dump(manifest, open(f'{ULT}/manifest.json', 'w'), indent=2, default=str)
print('Saved everything in', ULT)
for root, dirs, files in os.walk(ULT):
    depth = root[len(ULT):].count(os.sep)
    if depth <= 2 and 'restored' not in root:
        print('  ' * depth + os.path.basename(root) + '/', f'({len(files)} files)')